### Necessary Libraries


In [ ]:
!pip install torchmetrics
!pip install lpips


In [ ]:
import torch
from torchmetrics.image import StructuralSimilarityIndexMeasure
import torch
import lpips
from PIL import Image
from torchvision import transforms


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
import torchvision.utils as vutils
import matplotlib.animation as animation
from IPython.display import HTML

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd
import copy
import time
import cv2 as cv
from tqdm import tqdm_notebook as tqdm
import matplotlib.image as mpimg

### Encoder and Generator Model

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
IMG_WIDTH = 224
IMG_HEIGHT = 224
latent_size = 200

In [ ]:
!pip install linformer

In [ ]:
class GumbelQuantize(nn.Module):
    def __init__(self, num_hiddens, n_embed, embedding_dim, straight_through=False, kld_scale=5e-5, init_tau=1.0, min_tau=0.1, anneal_rate=0.00005):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.n_embed = n_embed
        self.straight_through = straight_through
        self.temperature = init_tau
        self.min_tau = min_tau
        self.anneal_rate = anneal_rate
        self.kld_scale = kld_scale
        self.proj = nn.Conv2d(256, n_embed, 1)
        self.embed = nn.Embedding(n_embed, embedding_dim)
        nn.init.orthogonal_(self.embed.weight)
        

    def forward(self, z):
        logits = self.proj(z)
        self.temperature = max(self.min_tau, self.temperature * (1 - self.anneal_rate))
        hard = self.straight_through if self.training else True
        soft_one_hot = F.gumbel_softmax(logits, tau=self.temperature, dim=1, hard=hard)
        ctx_indices = torch.zeros_like(soft_one_hot, dtype=torch.int64)
        cabac = CABAC()
        encoded = cabac.encode(soft_one_hot, ctx_indices)
        decoded = cabac.decode(encoded, ctx_indices)   
        z_q = torch.matmul(soft_one_hot.permute(0, 2, 3, 1), self.embed.weight).permute(0, 3, 1, 2)
        qy = F.softmax(logits, dim=1)
        diff = self.kld_scale * torch.sum(qy * torch.log(qy * self.n_embed + 1e-10), dim=1).mean()
        indices = soft_one_hot.argmax(dim=1)
        return z_q, diff


In [ ]:
import torch
import torch.nn as nn
from linformer import Linformer

class Encoder(nn.Module):
    def __init__(self, num_channels_in_encoder=8):  
        super(Encoder, self).__init__()
        self.e_conv_1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=(3, 3), stride=(2, 2), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),  
            nn.ReLU()
        )
        self.e_conv_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(2, 2), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),  
            nn.ReLU()
        )
        self.e_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.linformer_block = None
        self.e_conv_3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(2, 2), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1), 
            nn.ReLU()
        )
        self.gumbel_quant = GumbelQuantize(num_hiddens=64, n_embed=64 ,embedding_dim=8)
    def forward(self, x):
        ec1 = self.e_conv_1(x)  
        ec2 = self.e_conv_2(ec1)  
        eblock1 = self.e_block_1(ec2) + ec2 
        batch_size, channels, height, width = eblock1.shape
        seq_len = height * width  
        if self.linformer_block is None:
            self.linformer_block = Linformer(
                dim=64, seq_len=seq_len, depth=1, heads=4, k=64
            ).to(x.device)  
        eblock1_flat = eblock1.view(batch_size, seq_len, channels)  
        linform = self.linformer_block(eblock1_flat) 
        linform_reshaped = linform.view(batch_size, channels, height, width)
        ec3 = self.e_conv_3(linform_reshaped)  
        z_q, diff = self.gumbel_quant(ec3)
        return z_q, diff
        
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
netE = Encoder(num_channels_in_encoder=8).to(device)
inp = torch.randn(IMG_WIDTH * IMG_HEIGHT * 3 * 100).view((-1, 3, IMG_HEIGHT, IMG_WIDTH)).to(device)
output = netE(inp)[0]
print(output.shape)  
print('The Compression Ratio is :  ' + str((output.shape[1] * output.shape[2] * output.shape[3]) / (IMG_WIDTH * IMG_HEIGHT * 3) * 100))


In [ ]:
import torch
import torch.nn as nn

class WGAN_Generator(nn.Module):
    def __init__(self):
        super(WGAN_Generator, self).__init__()
        self.d_up_conv_1 = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=32, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=64, out_channels=64, kernel_size=(2, 2), stride=(2, 2)),  
            nn.ReLU(),
        )
        self.d_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.d_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.d_up_conv_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=128, out_channels=128, kernel_size=(2, 2), stride=(2, 2)),  
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=(2, 2), stride=(2, 2)),
            nn.ReLU(),
        )
        self.d_final_conv = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=3, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.Tanh()  
        )

    def forward(self, x):
        uc1 = self.d_up_conv_1(x)  
        dblock1 = self.d_block_1(uc1) + uc1  
        dblock2 = self.d_block_2(dblock1) + dblock1  
        uc2 = self.d_up_conv_2(dblock2)  
        dec = self.d_final_conv(uc2)  
        return dec


In [ ]:
netG = WGAN_Generator().to(device)
netG.apply(weights_init)
inp = torch.randn(100*num_channels_in_encoder*28*28).view((-1,num_channels_in_encoder,28,28)).to(device)
output = netG(inp)
print(output.shape)


In [ ]:
netE.load_state_dict(torch.load('/content/netE_epoch_3.pth'))
netG.load_state_dict(torch.load('/content/netG_epoch_3.pth'))
netG.eval()
netE.eval()


In [ ]:
##### make an directory with images/dummy/

In [ ]:
import torch
from torchmetrics.image import StructuralSimilarityIndexMeasure
ssim = StructuralSimilarityIndexMeasure(data_range=255.0).to(device)
ssim_value = ssim(input_tensor, reconstructed_tensor)
print(f"SSIM: {ssim_value.item():.4f}")


In [ ]:
lpips_score = torch.nn.functional.mse_loss(input_tensor, reconstructed_tensor)
print(f"LPIPS: {lpips_score.item():.4f}")

### CABAC Arithmetic Encoding and Decoding

In [ ]:
import torch

class CABAC:
    def __init__(self, num_contexts=64, precision=16):
        self.num_contexts = num_contexts
        self.context_probs = torch.full((num_contexts,), 0.5)  # Initialize context probabilities
        self.low = torch.tensor(0)
        self.high = torch.tensor((1 << precision) - 1)  # Full range
        self.scale = 1 << precision

    def _update_context(self, ctx_indices, bits):
        """
        Update context probabilities based on observed bits.
        Args:
            ctx_indices: Tensor of context indices.
            bits: Tensor of binary values (0 or 1).
        """
        learning_rate = 0.01
        ctx_indices = ctx_indices.view(-1)
        bits = bits.view(-1)
        for i in range(ctx_indices.size(0)):
            ctx_idx = ctx_indices[i]
            bit = bits[i]
            prob = self.context_probs[ctx_idx]
            if bit == 1:
                self.context_probs[ctx_idx] += learning_rate * (1 - prob)
            else:
                self.context_probs[ctx_idx] -= learning_rate * prob

    def encode(self, input_tensor, ctx_indices):
        """
        Encode a multi-dimensional tensor using CABAC.
        Args:
            input_tensor: Input tensor (binary values, 0 or 1).
            ctx_indices: Context indices tensor of the same shape.
        Returns:
            Encoded bitstream as a tensor.
        """
        original_shape = input_tensor.shape
        input_tensor = input_tensor.view(-1)
        ctx_indices = ctx_indices.view(-1)
        low = self.low.clone()
        high = self.high.clone()
        scale = self.scale
        bitstream = []

        for value, ctx_idx in zip(input_tensor, ctx_indices):
            prob = self.context_probs[ctx_idx]
            range_ = high - low + 1
            mid = low + (range_ * prob).floor()

            if value == 1:
                low = mid + 1
            else:
                high = mid

            while high < scale // 2 or low >= scale // 2:
                if high < scale // 2:
                    bitstream.append(0)
                    low *= 2
                    high = high * 2 + 1
                elif low >= scale // 2:
                    bitstream.append(1)
                    low -= scale // 2
                    high -= scale // 2
                low *= 2
                high = high * 2 + 1

            self._update_context(ctx_indices, input_tensor)

        return torch.tensor(bitstream, dtype=torch.int8)

    def decode(self, bitstream, ctx_indices):
        """
        Decode a bitstream using CABAC.
        Args:
            bitstream: Tensor of binary values (encoded stream).
            ctx_indices: Context indices tensor of the same shape as the original tensor.
        Returns:
            Decoded tensor matching the original input shape.
        """
        ctx_indices = ctx_indices.view(-1)
        low = self.low.clone()
        high = self.high.clone()
        scale = self.scale
        value = 0
        decoded = []

        for bit, ctx_idx in zip(bitstream, ctx_indices):
            prob = self.context_probs[ctx_idx]
            range_ = high - low + 1
            mid = low + (range_ * prob).floor()

            if value > mid:
                decoded.append(1)
                low = mid + 1
            else:
                decoded.append(0)
                high = mid

            while high < scale // 2 or low >= scale // 2:
                if high < scale // 2:
                    low *= 2
                    high = high * 2 + 1
                elif low >= scale // 2:
                    low -= scale // 2
                    high -= scale // 2
                    value -= scale // 2

                value = value * 2 + bit

            self._update_context(ctx_indices, torch.tensor(decoded[-1]).view(1))

        return torch.tensor(decoded, dtype=torch.int8).view(ctx_indices.size())



In [ ]:
batch_size, channels, height, width = 32, 8, 40, 40
input_tensor = torch.randint(0, 2, (batch_size, channels, height, width), dtype=torch.int8)
ctx_indices = torch.zeros_like(input_tensor, dtype=torch.int64)  # Example context indices
cabac = CABAC()
encoded = cabac.encode(input_tensor, ctx_indices)
print("Encoded Bitstream Shape:", encoded.shape)
decoded = cabac.decode(encoded, ctx_indices)
print("Decoded Tensor Shape:", decoded.shape)

    # Check for losslessness
assert torch.equal(input_tensor, decoded), "Decoded tensor does not match original!"
